### Imports

In [1]:
import tensorflow as tf
import numpy as np
from pathlib import Path

print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


### Paths

In [2]:
MODEL_PATH = "../models/fine_tuned_model.keras"

EXPORT_DIR = Path("../exports")
EXPORT_DIR.mkdir(exist_ok=True)

### Load Model

In [3]:
model = tf.keras.models.load_model(
    MODEL_PATH
)

print("Model Loaded Successfully")

Model Loaded Successfully


### Verify Model

In [4]:
model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)           │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ sequential_1 (Sequential)            │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ true_divide_1 (TrueDivide)           │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ subtract_1 (Subtract)                │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ mobilenetv2_1.00_224 (Functional)    │ (None, 7, 7, 1280)          │       2,257,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d_1           │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 1280)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 6)                   │           7,686 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 5,333,844 (20.35 MB)

 Trainable params: 1,534,086 (5.85 MB)

 Non-trainable params: 731,584 (2.79 MB)

 Optimizer params: 3,068,174 (11.70 MB)

### Convert to TensorFlow Lite

#### Standard TFLite Model

In [5]:
converter = tf.lite.TFLiteConverter.from_keras_model(
    model
)

tflite_model = converter.convert()

INFO:tensorflow:Assets written to: C:\Users\USER\AppData\Local\Temp\tmp8_8ptzrv\assets


INFO:tensorflow:Assets written to: C:\Users\USER\AppData\Local\Temp\tmp8_8ptzrv\assets


Saved artifact at 'C:\Users\USER\AppData\Local\Temp\tmp8_8ptzrv'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_4')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  1926738112208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1926738113360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1926738114128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1926738112016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1926738112400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1926738112592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1926738114896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1926738114704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1926738113552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1926738115472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  19267

#### Save Model

In [6]:
tflite_model_path = (
    EXPORT_DIR / "agrolens_model.tflite"
)

with open(
    tflite_model_path,
    "wb"
) as f:

    f.write(tflite_model)

print("TFLite model saved.")

TFLite model saved.


### Create Labels File

In [7]:
class_names = [
    "bacterial_leaf_blight",
    "blast",
    "brown_spot",
    "healthy",
    "leaf_scald",
    "sheath_blight"
]

In [8]:
with open(
    EXPORT_DIR / "labels.txt",
    "w"
) as f:

    for label in class_names:
        f.write(label + "\n")

print("Labels file saved.")

Labels file saved.


### Check File Size

In [9]:
size_mb = (
    tflite_model_path.stat().st_size
    / (1024 * 1024)
)

print(
    f"TFLite Size: {size_mb:.2f} MB"
)

TFLite Size: 8.50 MB


### Load TFLite Model

In [10]:
interpreter = tf.lite.Interpreter(
    model_path=str(tflite_model_path)
)

interpreter.allocate_tensors()

print("TFLite Model Loaded Successfully")

TFLite Model Loaded Successfully


D:\Programming\Flutter\AgroLens\ml\venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


### Inspect Input/Output

In [11]:
input_details = interpreter.get_input_details()

output_details = interpreter.get_output_details()

print(input_details)

print(output_details)

[{'name': 'serving_default_input_layer_4:0', 'index': 0, 'shape': array([  1, 224, 224,   3], dtype=int32), 'shape_signature': array([ -1, 224, 224,   3], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0, 'block_size': 0}, 'sparsity_parameters': {}}]
[{'name': 'StatefulPartitionedCall_1:0', 'index': 176, 'shape': array([1, 6], dtype=int32), 'shape_signature': array([-1,  6], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0, 'block_size': 0}, 'sparsity_parameters': {}}]


### Test Inference

In [12]:
dummy_input = np.random.random(
    (1,224,224,3)
).astype(np.float32)

In [13]:
interpreter.set_tensor(
    input_details[0]["index"],
    dummy_input
)

interpreter.invoke()

output = interpreter.get_tensor(
    output_details[0]["index"]
)

print(output)

[[0.00282065 0.08622306 0.5165664  0.06568938 0.3274314  0.00126915]]


### Optimized Version

In [14]:
converter = tf.lite.TFLiteConverter.from_keras_model(
    model
)

converter.optimizations = [
    tf.lite.Optimize.DEFAULT
]

optimized_tflite_model = converter.convert()

INFO:tensorflow:Assets written to: C:\Users\USER\AppData\Local\Temp\tmpm_cxw_45\assets


INFO:tensorflow:Assets written to: C:\Users\USER\AppData\Local\Temp\tmpm_cxw_45\assets


Saved artifact at 'C:\Users\USER\AppData\Local\Temp\tmpm_cxw_45'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_4')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  1926738112208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1926738113360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1926738114128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1926738112016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1926738112400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1926738112592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1926738114896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1926738114704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1926738113552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1926738115472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  19267

In [15]:
optimized_path = (
    EXPORT_DIR /
    "agrolens_model_optimized.tflite"
)

with open(
    optimized_path,
    "wb"
) as f:

    f.write(
        optimized_tflite_model
    )

In [16]:
size_mb = (
    optimized_path.stat().st_size
    / (1024 * 1024)
)

print(
    f"Optimized Size: {size_mb:.2f} MB"
)

Optimized Size: 2.42 MB


### Final Export Summary

In [17]:
print("="*50)

print("AGROLENS MODEL EXPORT COMPLETE")

print("="*50)

print("Model:")
print(optimized_path)

print("\nLabels:")
print(EXPORT_DIR / "labels.txt")

print("\nClasses:")
for label in class_names:
    print("-", label)

AGROLENS MODEL EXPORT COMPLETE
Model:
..\exports\agrolens_model_optimized.tflite

Labels:
..\exports\labels.txt

Classes:
- bacterial_leaf_blight
- blast
- brown_spot
- healthy
- leaf_scald
- sheath_blight
